# 02 - Integracion de datos a nivel de persona

Integra las fuentes de `data/processed/` en tablas agregadas a nivel `IDPERSONA`
(1 fila = 1 persona) y produce `data/features/dataset_personas_integrado.csv`.

**Regla principal:** nunca hacer JOIN directo entre tablas de detalle 1:N.
Cada fuente se agrega primero a nivel persona y luego se integra con LEFT JOIN
sobre una tabla base de personas.

**No se hace en este notebook:** preprocesamiento (ya esta en `data/processed/`),
clustering, embeddings, dashboard, ni seleccion final de variables para ML.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
ENCODINGS = ("utf-8-sig", "cp1252", "latin-1")

## Funciones auxiliares reutilizables

In [2]:
def leer_csv(nombre_archivo: str, **kwargs) -> pd.DataFrame:
    """Lee un CSV de data/processed probando distintas codificaciones."""
    ruta = PROCESSED_DIR / nombre_archivo
    ultimo_error = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(ruta, encoding=enc, low_memory=False, **kwargs)
            print(f"Leido {nombre_archivo}: {df.shape[0]} filas x {df.shape[1]} columnas")
            return df
        except (UnicodeDecodeError, UnicodeError) as e:
            ultimo_error = e
    raise ultimo_error


def validar_llave(df: pd.DataFrame, llave: str, nombre: str) -> None:
    """Imprime diagnostico de una tabla de detalle antes de agregarla/unirla."""
    dups = df[llave].duplicated().sum()
    print(f"[{nombre}] llave={llave} | filas={len(df)} | "
          f"personas={df[llave].nunique(dropna=True)} | duplicados_llave={dups}")


def verificar_unicidad(df: pd.DataFrame, llave: str) -> bool:
    """True si `llave` identifica de forma unica cada fila de df."""
    return df[llave].is_unique


def validar_join(antes: pd.DataFrame, despues: pd.DataFrame, llave: str, nombre: str = "") -> None:
    """Compara filas/personas antes y despues de un LEFT JOIN sobre `llave`."""
    dup_despues = despues[llave].duplicated().sum()
    print(f"[JOIN {nombre}] filas: {len(antes)} -> {len(despues)} | "
          f"personas: {antes[llave].nunique()} -> {despues[llave].nunique()} | "
          f"duplicados_llave_despues={dup_despues}")
    if len(despues) != len(antes):
        print(f"  ADVERTENCIA: el numero de filas cambio tras el join ({nombre})")
    if dup_despues:
        print(f"  ADVERTENCIA: {llave} quedo duplicado tras el join ({nombre})")


def guardar_tabla(df: pd.DataFrame, nombre_archivo: str, llave: str = "IDPERSONA") -> Path:
    """Valida unicidad de `llave` y guarda la tabla en data/features/."""
    assert verificar_unicidad(df, llave), f"{nombre_archivo}: '{llave}' no es unico"
    ruta = FEATURES_DIR / nombre_archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Guardado {ruta.name}: {df.shape[0]} filas x {df.shape[1]} columnas")
    return ruta


def cobertura(numerador: pd.Series, denominador: pd.Series) -> pd.Series:
    """numerador / denominador elemento a elemento, evitando division por cero (-> NA)."""
    denom = denominador.replace(0, np.nan)
    return (numerador / denom).round(4)

## Carga de fuentes crudas (data/processed)

In [3]:
titulaciones = leer_csv("reporte_titulaciones_educacion.csv").rename(columns={"IdPersona": "IDPERSONA"})
capacitaciones = leer_csv("capacitaciones_todas.csv")
certificados = leer_csv("certificados_todos.csv")
ponentes = leer_csv("ponentes_todos.csv")
carga_academica = leer_csv("carga_academica_disponible.csv")
carga_politecnica = leer_csv("carga_politecnica_disponible.csv")
catalogo_actividades = leer_csv("catalogo_actividades_carga.csv")
catalogo_idiomas = leer_csv("catalogo_idiomas.csv")
datos_personales = leer_csv("datos_personales_ultimos_5anios.csv")
experiencia_externa = leer_csv("experiencia_externa.csv")
heteroevaluacion = leer_csv("heteroevaluacion_disponible.csv")
idiomas_personas = leer_csv("idiomas_personas.csv")
mencion_honor = leer_csv("mencion_honor.csv")
proyecto_grado = leer_csv("proyecto_grado.csv")
proyectos_investigacion = leer_csv("proyectos_investigacion_disponible.csv")
proyectos_vinculacion = leer_csv("proyectos_vinculacion_disponible.csv")
publicaciones = leer_csv("publicaciones.csv")
historial_laboral_features = leer_csv("historial_laboral_features.csv")

Leido reporte_titulaciones_educacion.csv: 74553 filas x 30 columnas


Leido capacitaciones_todas.csv: 64563 filas x 30 columnas
Leido certificados_todos.csv: 5096 filas x 20 columnas
Leido ponentes_todos.csv: 1257 filas x 31 columnas
Leido carga_academica_disponible.csv: 43701 filas x 32 columnas


Leido carga_politecnica_disponible.csv: 48709 filas x 32 columnas
Leido catalogo_actividades_carga.csv: 110 filas x 10 columnas
Leido catalogo_idiomas.csv: 4 filas x 2 columnas
Leido datos_personales_ultimos_5anios.csv: 2213 filas x 8 columnas
Leido experiencia_externa.csv: 13818 filas x 20 columnas
Leido heteroevaluacion_disponible.csv: 76644 filas x 16 columnas
Leido idiomas_personas.csv: 4137 filas x 9 columnas
Leido mencion_honor.csv: 5457 filas x 10 columnas
Leido proyecto_grado.csv: 7971 filas x 7 columnas


Leido proyectos_investigacion_disponible.csv: 6768 filas x 44 columnas
Leido proyectos_vinculacion_disponible.csv: 2659 filas x 10 columnas
Leido publicaciones.csv: 8757 filas x 26 columnas
Leido historial_laboral_features.csv: 3951 filas x 31 columnas


## Relaciones ambiguas: no inventar joins

Antes de integrar, se revisaron 3 relaciones que no son evidentes y **no se
resuelven de forma automatica**:

| TABLA | LLAVE DISPONIBLE | PROBLEMA | DECISION RECOMENDADA |
|---|---|---|---|
| `catalogo_actividades_carga.csv` | `IDTIPOACTIVIDAD` (unico en el catalogo) | Solo ~7% de los `IDTIPOACTIVIDAD` de `carga_politecnica_disponible.csv` existen en el catalogo: son espacios de codigos distintos, no la misma clasificacion. | No unir. Se usa `carga_politecnica_disponible.csv` con sus propias columnas (`APLICA_T1/T2/T3`, horas) sin decodificar `IDTIPOACTIVIDAD`. |
| `catalogo_idiomas.csv` | `CODIGOSTR` (N/B/I/A) | No comparte llave con `IDIOMA`/`IDIDIOMA` de `idiomas_personas.csv`. Sus codigos (N/B/I/A) en realidad describen los niveles `NIVELLECTURA`/`NIVELESCRITURA`/`NIVELCONVERSACION`, no el idioma en si. | No se fusiona en `idiomas_persona.csv` (no aporta variable a nivel persona); queda documentado como diccionario de niveles, no de idiomas. |
| `proyecto_grado.csv` | `IDDIRECTOR` (sin columna `IDPERSONA` explicita) | Se verifico solapamiento de `IDDIRECTOR` contra la poblacion base (`historial_laboral_features` + `datos_personales`): ~70% (590/841) coincide. El resto puede ser personal fuera de la ventana de poblacion o datos externos. | Se asume `IDDIRECTOR == IDPERSONA` (unica llave disponible) y se renombra la columna. Los directores sin correspondencia quedaran como filas no encontradas en el LEFT JOIN final (no se inventa una persona). |

Ver detalle de la verificacion de `proyecto_grado` en la celda siguiente.

In [4]:
# Verificacion de cobertura IDDIRECTOR vs poblacion base
poblacion_base_ids = set(historial_laboral_features["IDPERSONA"]) | set(datos_personales["IDPERSONA"])
directores_ids = set(proyecto_grado["IDDIRECTOR"].dropna().unique())
coincidencias = directores_ids & poblacion_base_ids
print(f"IDDIRECTOR unicos: {len(directores_ids)} | coinciden con poblacion base: {len(coincidencias)} "
      f"({len(coincidencias) / len(directores_ids):.1%})")

IDDIRECTOR unicos: 841 | coinciden con poblacion base: 590 (70.2%)


## Tabla base de personas

La poblacion se define como la union de `IDPERSONA` presentes en
`historial_laboral_features.csv` (tabla de features de historial laboral, ya
a nivel persona) y en `datos_personales_ultimos_5anios.csv` (para no perder
personas con datos personales pero sin historial laboral). Sobre esa union se
integran despues todas las demas tablas agregadas.

In [5]:
validar_llave(datos_personales, "IDPERSONA", "datos_personales_ultimos_5anios")
validar_llave(historial_laboral_features, "IDPERSONA", "historial_laboral_features")

ids_base = pd.Index(
    sorted(set(historial_laboral_features["IDPERSONA"]) | set(datos_personales["IDPERSONA"]))
)
personas = pd.DataFrame({"IDPERSONA": ids_base})
personas = personas.merge(datos_personales, on="IDPERSONA", how="left")
validar_join(pd.DataFrame({"IDPERSONA": ids_base}), personas, "IDPERSONA", "personas + datos_personales")

guardar_tabla(personas, "personas.csv")
personas.head()

[datos_personales_ultimos_5anios] llave=IDPERSONA | filas=2213 | personas=2213 | duplicados_llave=0
[historial_laboral_features] llave=IDPERSONA | filas=3951 | personas=3951 | duplicados_llave=0
[JOIN personas + datos_personales] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
Guardado personas.csv: 3956 filas x 8 columnas


,IDPERSONA,ESTADOCIVIL,FECHANACIMIENTO,SEXO,PAISORIGEN,PROVINCIAORIGEN,CANTONORIGEN,EDAD
0,26,C,1972-01-01,M,ECUADOR,EL ORO,PIÑAS,54.0
1,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Titulaciones (formacion academica)

Fuente: `reporte_titulaciones_educacion.csv`. Llave `IdPersona` (renombrada a
`IDPERSONA`). Puede tener varias titulaciones por persona; se agrega por
`IDPERSONA` y `IdTitulacion` para eliminar duplicados exactos antes de contar.

In [6]:
validar_llave(titulaciones, "IDPERSONA", "reporte_titulaciones_educacion")

titulaciones_dedup = titulaciones.drop_duplicates(subset=["IDPERSONA", "IdTitulacion"])

niveles_dummy = pd.get_dummies(titulaciones_dedup["Nivel"], prefix="NUM_TIT")
niveles_dummy.columns = niveles_dummy.columns.str.replace(" ", "_")
niveles_por_persona = pd.concat([titulaciones_dedup["IDPERSONA"], niveles_dummy], axis=1).groupby("IDPERSONA").sum()

titulaciones_persona = (
    titulaciones_dedup.groupby("IDPERSONA")
    .agg(NUM_TITULACIONES=("IdTitulacion", "nunique"))
    .join(niveles_por_persona)
    .reset_index()
)
titulaciones_persona["TIENE_POSGRADO"] = (titulaciones_persona.get("NUM_TIT_CUARTO_NIVEL", 0) > 0).astype(int)

guardar_tabla(titulaciones_persona, "titulaciones_persona.csv")
titulaciones_persona.head()

[reporte_titulaciones_educacion] llave=IDPERSONA | filas=74553 | personas=61658 | duplicados_llave=12895


Guardado titulaciones_persona.csv: 61658 filas x 7 columnas


,IDPERSONA,NUM_TITULACIONES,NUM_TIT_BACHILLERATO,NUM_TIT_CUARTO_NIVEL,NUM_TIT_PRIMARIA,NUM_TIT_TERCER_NIVEL,TIENE_POSGRADO
0,16,1,0,1,0,0,1
1,17,1,0,0,0,1,0
2,18,1,0,0,0,1,0
3,19,3,0,2,0,1,1
4,21,1,0,1,0,0,1


## Capacitaciones

Fuente: `capacitaciones_todas.csv`. Se agrega por separado de certificaciones
y ponencias aunque compartan estructura, por ser conceptualmente distintas.

In [7]:
validar_llave(capacitaciones, "IDPERSONA", "capacitaciones_todas")

capacitaciones_persona = capacitaciones.groupby("IDPERSONA").agg(
    NUM_CAPACITACIONES=("IDCAPACITACION", "nunique"),
    SUMA_DURACION_CAPACITACIONES=("DURACION", "sum"),
    NUM_CAPACITACIONES_APROBACION=("TIPODESCRIPCION", lambda s: (s == "APROBACION").sum()),
    NUM_CAPACITACIONES_VIRTUAL=("TIPOMODALIDADDESCRIPCION", lambda s: (s == "VIRTUAL").sum()),
    NUM_CAPACITACIONES_PRESENCIAL=("TIPOMODALIDADDESCRIPCION", lambda s: (s == "PRESENCIAL").sum()),
).reset_index()

guardar_tabla(capacitaciones_persona, "capacitaciones_persona.csv")
capacitaciones_persona.head()

[capacitaciones_todas] llave=IDPERSONA | filas=64563 | personas=4599 | duplicados_llave=59964


Guardado capacitaciones_persona.csv: 4599 filas x 6 columnas


,IDPERSONA,NUM_CAPACITACIONES,SUMA_DURACION_CAPACITACIONES,NUM_CAPACITACIONES_APROBACION,NUM_CAPACITACIONES_VIRTUAL,NUM_CAPACITACIONES_PRESENCIAL
0,26,12,590.0,6,1,0
1,38,4,606.0,2,0,3
2,41,14,126.0,4,1,0
3,55,4,40.0,0,0,2
4,57,1,8.0,0,0,0


## Certificaciones

Fuente: `certificados_todos.csv`.

In [8]:
validar_llave(certificados, "IDPERSONA", "certificados_todos")

certificaciones_persona = certificados.groupby("IDPERSONA").agg(
    NUM_CERTIFICACIONES=("IDCAPACITACION", "nunique"),
    SUMA_DURACION_CERTIFICACIONES=("DURACION", "sum"),
).reset_index()

guardar_tabla(certificaciones_persona, "certificaciones_persona.csv")
certificaciones_persona.head()

[certificados_todos] llave=IDPERSONA | filas=5096 | personas=1772 | duplicados_llave=3324
Guardado certificaciones_persona.csv: 1772 filas x 3 columnas


,IDPERSONA,NUM_CERTIFICACIONES,SUMA_DURACION_CERTIFICACIONES
0,102,2,122.0
1,228,1,68.0
2,229,3,3.0
3,231,4,43.0
4,697,8,91.0


## Ponencias

Fuente: `ponentes_todos.csv`.

In [9]:
validar_llave(ponentes, "IDPERSONA", "ponentes_todos")

ponencias_persona = ponentes.groupby("IDPERSONA").agg(
    NUM_PONENCIAS=("IDCAPACITACION", "nunique"),
    SUMA_DURACION_PONENCIAS=("DURACION", "sum"),
).reset_index()

guardar_tabla(ponencias_persona, "ponencias_persona.csv")
ponencias_persona.head()

[ponentes_todos] llave=IDPERSONA | filas=1257 | personas=530 | duplicados_llave=727
Guardado ponencias_persona.csv: 530 filas x 3 columnas


,IDPERSONA,NUM_PONENCIAS,SUMA_DURACION_PONENCIAS
0,26,3,25.0
1,62,1,4.0
2,76,2,16.0
3,86,1,8.0
4,89,1,0.0


## Docencia (carga academica)

Fuente: `carga_academica_disponible.csv`. Una fila por curso/paralelo/periodo
asignado a un docente; se agrega por `IDPERSONA`.

In [10]:
validar_llave(carga_academica, "IDPERSONA", "carga_academica_disponible")

docencia_persona = carga_academica.groupby("IDPERSONA").agg(
    NUM_CURSOS=("IDCURSO", "nunique"),
    NUM_PERIODOS_DOCENCIA=("IDPERIODO", "nunique"),
    NUM_ASIGNACIONES_DOCENCIA=("IDCPLCAMBIOSCURSO", "nunique"),
    TOTAL_HORAS_DOCENCIA=("TOTALHORAS", "sum"),
    TOTAL_ESTUDIANTES=("NUMREGISTRADOS", "sum"),
    PROMEDIO_ESTUDIANTES_POR_CURSO=("NUMREGISTRADOS", "mean"),
).reset_index()
docencia_persona["PROMEDIO_ESTUDIANTES_POR_CURSO"] = docencia_persona["PROMEDIO_ESTUDIANTES_POR_CURSO"].round(2)

guardar_tabla(docencia_persona, "docencia_persona.csv")
docencia_persona.head()

[carga_academica_disponible] llave=IDPERSONA | filas=43701 | personas=1141 | duplicados_llave=42560
Guardado docencia_persona.csv: 1141 filas x 7 columnas


,IDPERSONA,NUM_CURSOS,NUM_PERIODOS_DOCENCIA,NUM_ASIGNACIONES_DOCENCIA,TOTAL_HORAS_DOCENCIA,TOTAL_ESTUDIANTES,PROMEDIO_ESTUDIANTES_POR_CURSO
0,26,35,14,35,3164.45,639,18.26
1,38,12,6,12,1211.73,166,13.83
2,40,27,8,27,1825.58,206,7.63
3,86,42,14,42,5288.30,1132,26.95
4,94,30,14,30,4024.95,882,29.40


## Carga politecnica

Fuente: `carga_politecnica_disponible.csv`. Ya trae `APLICA_T1/T2/T3`,
`NUM_TERMINOS` y `TERMINOS_ACTIVOS` por actividad (no se recalculan). Se
agregan a nivel persona sumando/contando esas columnas existentes.
`NUMHORASCP` se usa como horas de carga politecnica (mismo orden de magnitud
que las cargas reportadas; `NUMHORAS` es una columna auxiliar mas pequeÃ±a).

In [11]:
validar_llave(carga_politecnica, "IDPERSONA", "carga_politecnica_disponible")

carga_politecnica_persona = carga_politecnica.groupby("IDPERSONA").agg(
    NUM_ACTIVIDADES_POLITECNICAS=("IDACTIVIDAD", "nunique"),
    NUM_ACTIVIDADES_T1=("APLICA_T1", "sum"),
    NUM_ACTIVIDADES_T2=("APLICA_T2", "sum"),
    NUM_ACTIVIDADES_T3=("APLICA_T3", "sum"),
    TOTAL_HORAS_POLITECNICAS=("NUMHORASCP", "sum"),
).reset_index()

guardar_tabla(carga_politecnica_persona, "carga_politecnica_persona.csv")
carga_politecnica_persona.head()

[carga_politecnica_disponible] llave=IDPERSONA | filas=48709 | personas=2481 | duplicados_llave=46228
Guardado carga_politecnica_persona.csv: 2481 filas x 6 columnas


,IDPERSONA,NUM_ACTIVIDADES_POLITECNICAS,NUM_ACTIVIDADES_T1,NUM_ACTIVIDADES_T2,NUM_ACTIVIDADES_T3,TOTAL_HORAS_POLITECNICAS
0,19,1,1,0,0,60
1,26,74,63,69,34,21791
2,38,32,27,28,7,6302
3,40,19,18,18,13,4512
4,58,4,4,0,0,624


## Experiencia externa

Fuente: `experiencia_externa.csv`. Se usan las columnas ya decodificadas
`CATEXPERIENCIA_DESC` y `ROLACADEMICO_DESC`.

In [12]:
validar_llave(experiencia_externa, "IDPERSONA", "experiencia_externa")

experiencia_externa_persona = experiencia_externa.groupby("IDPERSONA").agg(
    NUM_EXPERIENCIAS_EXTERNAS=("IDHISTORIALABORAL", "nunique"),
    NUM_EXPERIENCIAS_ACADEMICAS=("CATEXPERIENCIA_DESC", lambda s: (s == "ACADEMICA").sum()),
    NUM_EXPERIENCIAS_ADMINISTRATIVAS=("CATEXPERIENCIA_DESC", lambda s: (s == "ADMINISTRATIVA").sum()),
    NUM_EXPERIENCIAS_POR_CLASIFICAR=("CATEXPERIENCIA_DESC", lambda s: (s == "POR CLASIFICAR").sum()),
    NUM_EXPERIENCIAS_ROL_PROFESOR=("ROLACADEMICO_DESC", lambda s: (s == "PROFESOR").sum()),
).reset_index()

guardar_tabla(experiencia_externa_persona, "experiencia_externa_persona.csv")
experiencia_externa_persona.head()

[experiencia_externa] llave=IDPERSONA | filas=13818 | personas=4091 | duplicados_llave=9727


Guardado experiencia_externa_persona.csv: 4091 filas x 6 columnas


,IDPERSONA,NUM_EXPERIENCIAS_EXTERNAS,NUM_EXPERIENCIAS_ACADEMICAS,NUM_EXPERIENCIAS_ADMINISTRATIVAS,NUM_EXPERIENCIAS_POR_CLASIFICAR,NUM_EXPERIENCIAS_ROL_PROFESOR
0,26,3,1,0,2,0
1,40,2,0,0,2,0
2,41,2,0,0,2,0
3,58,2,0,0,2,0
4,86,3,0,0,3,0


## Idiomas

Fuente: `idiomas_personas.csv`. No se convierte `NIVELMCER` (A1-C2) a numero
todavia (queda para la etapa de preparacion para clustering). `catalogo_idiomas.csv`
no se fusiona aqui (ver seccion de relaciones ambiguas).

In [13]:
validar_llave(idiomas_personas, "IDPERSONA", "idiomas_personas")

idiomas_persona = idiomas_personas.groupby("IDPERSONA").agg(
    NUM_IDIOMAS=("IDIDIOMA", "nunique"),
    NUM_IDIOMAS_NO_NATIVOS=("LENGUANATIVA", lambda s: (s == 0).sum()),
    NUM_IDIOMAS_CON_NIVELMCER=("NIVELMCER", "count"),
).reset_index()

guardar_tabla(idiomas_persona, "idiomas_persona.csv")
idiomas_persona.head()

[idiomas_personas] llave=IDPERSONA | filas=4137 | personas=2381 | duplicados_llave=1756
Guardado idiomas_persona.csv: 2381 filas x 4 columnas


,IDPERSONA,NUM_IDIOMAS,NUM_IDIOMAS_NO_NATIVOS,NUM_IDIOMAS_CON_NIVELMCER
0,26,1,1,0
1,38,2,0,0
2,86,1,1,0
3,94,2,2,0
4,96,1,1,0


## Heteroevaluacion docente

Fuente: `heteroevaluacion_disponible.csv`. Una fila por curso/periodo
evaluado; se agrega **primero por `IDPERSONA`** (no se une fila a fila con
`carga_academica_disponible.csv`).

In [14]:
validar_llave(heteroevaluacion, "IDPERSONA", "heteroevaluacion_disponible")

evaluacion_persona = heteroevaluacion.groupby("IDPERSONA").agg(
    NUM_EVALUACIONES=("IDCURSO", "count"),
    PROMEDIO_HETEROEVALUACION=("PROMEDIO", "mean"),
    TOTAL_REGISTRADOS=("REGISTRADOS", "sum"),
    TOTAL_EVALUADOS=("EVALUADOS", "sum"),
).reset_index()
evaluacion_persona["PROMEDIO_HETEROEVALUACION"] = evaluacion_persona["PROMEDIO_HETEROEVALUACION"].round(2)
evaluacion_persona["COBERTURA_EVALUACION"] = cobertura(
    evaluacion_persona["TOTAL_EVALUADOS"], evaluacion_persona["TOTAL_REGISTRADOS"]
)

guardar_tabla(evaluacion_persona, "evaluacion_persona.csv")
evaluacion_persona.head()

[heteroevaluacion_disponible] llave=IDPERSONA | filas=76644 | personas=2516 | duplicados_llave=74127
Guardado evaluacion_persona.csv: 2516 filas x 6 columnas


,IDPERSONA,NUM_EVALUACIONES,PROMEDIO_HETEROEVALUACION,TOTAL_REGISTRADOS,TOTAL_EVALUADOS,COBERTURA_EVALUACION
0,16.0,0,86.50,870,707,0.8126
1,18.0,0,84.51,395,303,0.7671
2,19.0,0,74.48,1021,846,0.8286
3,24.0,0,92.54,24,16,0.6667
4,26.0,26,86.02,1047,886,0.8462


## Reconocimientos (menciones de honor)

Fuente: `mencion_honor.csv`.

In [15]:
validar_llave(mencion_honor, "IDPERSONA", "mencion_honor")

reconocimientos_persona = mencion_honor.groupby("IDPERSONA").agg(
    NUM_RECONOCIMIENTOS=("IDMENCIONHONOR", "nunique"),
    NUM_TIPOS_RECONOCIMIENTO_DISTINTOS=("TIPO", "nunique"),
).reset_index()

guardar_tabla(reconocimientos_persona, "reconocimientos_persona.csv")
reconocimientos_persona.head()

[mencion_honor] llave=IDPERSONA | filas=5457 | personas=1647 | duplicados_llave=3810
Guardado reconocimientos_persona.csv: 1647 filas x 3 columnas


,IDPERSONA,NUM_RECONOCIMIENTOS,NUM_TIPOS_RECONOCIMIENTO_DISTINTOS
0,26,2,0
1,38,2,0
2,41,1,0
3,54,4,0
4,58,5,0


## Direccion de trabajos de titulacion (proyecto de grado)

Fuente: `proyecto_grado.csv`. Solo trae `IDDIRECTOR` (ver decision documentada
arriba): se renombra a `IDPERSONA` antes de agregar.

In [16]:
proyecto_grado_persona_src = proyecto_grado.rename(columns={"IDDIRECTOR": "IDPERSONA"}).dropna(subset=["IDPERSONA"])
proyecto_grado_persona_src["IDPERSONA"] = proyecto_grado_persona_src["IDPERSONA"].astype("int64")
validar_llave(proyecto_grado_persona_src, "IDPERSONA", "proyecto_grado (IDDIRECTOR->IDPERSONA)")

proyecto_grado_persona = proyecto_grado_persona_src.groupby("IDPERSONA").agg(
    NUM_PROYECTOS_GRADO_DIRIGIDOS=("NOMBRETRABAJOTITULACION", "count"),
    NUM_PROGRAMAS_TITULACION_DISTINTOS=("NOMBREPROGRAMA", "nunique"),
).reset_index()

guardar_tabla(proyecto_grado_persona, "proyecto_grado_persona.csv")
proyecto_grado_persona.head()

[proyecto_grado (IDDIRECTOR->IDPERSONA)] llave=IDPERSONA | filas=7720 | personas=841 | duplicados_llave=6879
Guardado proyecto_grado_persona.csv: 841 filas x 3 columnas


,IDPERSONA,NUM_PROYECTOS_GRADO_DIRIGIDOS,NUM_PROGRAMAS_TITULACION_DISTINTOS
0,0,15,4
1,16,2,1
2,19,2,2
3,26,7,1
4,62,1,1


## Proyectos de investigacion

Fuente: `proyectos_investigacion_disponible.csv`.

In [17]:
validar_llave(proyectos_investigacion, "IDPERSONA", "proyectos_investigacion_disponible")

investigacion_persona = proyectos_investigacion.groupby("IDPERSONA").agg(
    NUM_PROYECTOS_INVESTIGACION=("IDPROYECTOINVESTIGACION", "nunique"),
    NUM_PROYECTOS_ACTIVOS=("ESTADO_PROYECTO", lambda s: s.str.contains("EJECUCI", na=False).sum()),
    NUM_PROYECTOS_FINALIZADOS=("ESTADO_PROYECTO", lambda s: s.isin(["FINALIZADO", "CERRADO"]).sum()),
    NUM_PROYECTOS_COMO_DIRECTOR=("ROLPROYECTO", lambda s: (s == "DIRECTOR").sum()),
    NUM_PROYECTOS_COMO_CODIRECTOR=("ROLPROYECTO", lambda s: (s == "CO-DIRECTOR").sum()),
    NUM_PROYECTOS_COMO_PARTICIPANTE=("ROLPROYECTO", lambda s: (s == "PARTICIPANTE").sum()),
).reset_index()

guardar_tabla(investigacion_persona, "investigacion_persona.csv")
investigacion_persona.head()

[proyectos_investigacion_disponible] llave=IDPERSONA | filas=6768 | personas=1230 | duplicados_llave=5538

Guardado investigacion_persona.csv: 1230 filas x 7 columnas


,IDPERSONA,NUM_PROYECTOS_INVESTIGACION,NUM_PROYECTOS_ACTIVOS,NUM_PROYECTOS_FINALIZADOS,NUM_PROYECTOS_COMO_DIRECTOR,NUM_PROYECTOS_COMO_CODIRECTOR,NUM_PROYECTOS_COMO_PARTICIPANTE
0,26,24,11,15,9,4,13
1,38,2,2,0,0,0,2
2,86,5,2,3,0,0,5
3,94,11,6,6,7,1,4
4,96,7,3,4,5,0,2


## Proyectos de vinculacion

Fuente: `proyectos_vinculacion_disponible.csv`.

In [18]:
validar_llave(proyectos_vinculacion, "IDPERSONA", "proyectos_vinculacion_disponible")

vinculacion_persona = proyectos_vinculacion.groupby("IDPERSONA").agg(
    NUM_PROYECTOS_VINCULACION=("IDPROYECTOINVESTIGACION", "nunique"),
    NUM_VINCULACION_TUTOR=("ROLPROYECTO", lambda s: (s == "TUTOR").sum()),
    NUM_VINCULACION_DIRECTOR_PROYECTO=("ROLPROYECTO", lambda s: (s == "DIRECTOR DE PROYECTO").sum()),
    NUM_VINCULACION_DIRECTOR_PROGRAMA=("ROLPROYECTO", lambda s: (s == "DIRECTOR DE PROGRAMA").sum()),
).reset_index()

guardar_tabla(vinculacion_persona, "vinculacion_persona.csv")
vinculacion_persona.head()

[proyectos_vinculacion_disponible] llave=IDPERSONA | filas=2659 | personas=453 | duplicados_llave=2206
Guardado vinculacion_persona.csv: 453 filas x 5 columnas


,IDPERSONA,NUM_PROYECTOS_VINCULACION,NUM_VINCULACION_TUTOR,NUM_VINCULACION_DIRECTOR_PROYECTO,NUM_VINCULACION_DIRECTOR_PROGRAMA
0,40,4,4,0,0
1,86,1,1,0,0
2,129,7,2,5,0
3,398,1,1,0,0
4,419,1,1,0,0


## Publicaciones

Fuente: `publicaciones.csv`. Se usan directamente las columnas ya
codificadas (`REVISTAINDEXADA`, `REVISIONPARES`, `CUARTIL`, `CUARTILCITESCORE`).

In [19]:
validar_llave(publicaciones, "IDPERSONA", "publicaciones")

publicaciones_persona = publicaciones.groupby("IDPERSONA").agg(
    NUM_PUBLICACIONES=("IDPUBLICACIONPERSONA", "nunique"),
    NUM_PUBLICACIONES_INDEXADAS=("REVISTAINDEXADA", lambda s: (s == 1).sum()),
    NUM_PUBLICACIONES_REVISION_PARES=("REVISIONPARES", lambda s: (s == 1).sum()),
    NUM_PUBLICACIONES_Q1=("CUARTIL", lambda s: (s == "Q1").sum()),
    NUM_PUBLICACIONES_Q2=("CUARTIL", lambda s: (s == "Q2").sum()),
).reset_index()

guardar_tabla(publicaciones_persona, "publicaciones_persona.csv")
publicaciones_persona.head()

[publicaciones] llave=IDPERSONA | filas=8757 | personas=942 | duplicados_llave=7815


Guardado publicaciones_persona.csv: 942 filas x 6 columnas

,IDPERSONA,NUM_PUBLICACIONES,NUM_PUBLICACIONES_INDEXADAS,NUM_PUBLICACIONES_REVISION_PARES,NUM_PUBLICACIONES_Q1,NUM_PUBLICACIONES_Q2
0,26,53,37,0,11,4
1,38,2,0,0,2,0
2,86,4,3,0,0,0
3,94,13,11,0,4,3
4,96,25,9,10,6,0


## Historial laboral (ya agregado)

`historial_laboral_features.csv` ya fue construido en `10_historial_laboral_personas.ipynb`
a nivel de persona. Aqui solo se valida su unicidad; **no se vuelve a agregar**
ni se crea `historial_laboral_persona.csv`.

In [20]:
validar_llave(historial_laboral_features, "IDPERSONA", "historial_laboral_features")
assert verificar_unicidad(historial_laboral_features, "IDPERSONA"), "IDPERSONA no es unico en historial_laboral_features"
print("historial_laboral_features OK: IDPERSONA es unico")

[historial_laboral_features] llave=IDPERSONA | filas=3951 | personas=3951 | duplicados_llave=0
historial_laboral_features OK: IDPERSONA es unico


## Integracion final

LEFT JOIN de todas las tablas agregadas sobre la tabla base `personas`. Se
valida numero de filas/personas y duplicados de `IDPERSONA` despues de cada
join.

In [21]:
tablas_a_integrar = [
    ("titulaciones_persona", titulaciones_persona),
    ("capacitaciones_persona", capacitaciones_persona),
    ("certificaciones_persona", certificaciones_persona),
    ("ponencias_persona", ponencias_persona),
    ("docencia_persona", docencia_persona),
    ("carga_politecnica_persona", carga_politecnica_persona),
    ("experiencia_externa_persona", experiencia_externa_persona),
    ("idiomas_persona", idiomas_persona),
    ("evaluacion_persona", evaluacion_persona),
    ("reconocimientos_persona", reconocimientos_persona),
    ("proyecto_grado_persona", proyecto_grado_persona),
    ("investigacion_persona", investigacion_persona),
    ("vinculacion_persona", vinculacion_persona),
    ("publicaciones_persona", publicaciones_persona),
    ("historial_laboral_features", historial_laboral_features),
]

dataset_personas_integrado = personas.copy()
for nombre, tabla in tablas_a_integrar:
    assert verificar_unicidad(tabla, "IDPERSONA"), f"{nombre}: IDPERSONA no es unico, no se puede unir"
    antes = dataset_personas_integrado.copy()
    dataset_personas_integrado = dataset_personas_integrado.merge(tabla, on="IDPERSONA", how="left")
    validar_join(antes, dataset_personas_integrado, "IDPERSONA", nombre)

# Contadores (NUM_*) se rellenan con 0 cuando la persona no aparece en la fuente
columnas_conteo = [c for c in dataset_personas_integrado.columns if c.startswith(("NUM_", "TOTAL_", "SUMA_"))]
dataset_personas_integrado[columnas_conteo] = dataset_personas_integrado[columnas_conteo].fillna(0)

[JOIN titulaciones_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN capacitaciones_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN certificaciones_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN ponencias_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN docencia_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN carga_politecnica_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN experiencia_externa_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN idiomas_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN evaluacion_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | duplicados_llave_despues=0
[JOIN reconocimientos_persona] filas: 3956 -> 3956 | personas: 3956 -> 3956 | d

## Validacion final

In [22]:
assert dataset_personas_integrado["IDPERSONA"].is_unique, "IDPERSONA no es unico en el dataset final"

print(f"Personas totales: {dataset_personas_integrado['IDPERSONA'].nunique()}")
print(f"Columnas totales: {dataset_personas_integrado.shape[1]}")
print(f"Duplicados de IDPERSONA: {dataset_personas_integrado['IDPERSONA'].duplicated().sum()}")

nulos_pct = (dataset_personas_integrado.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nPorcentaje de nulos por columna (top 20):")
print(nulos_pct.head(20))

Personas totales: 3956
Columnas totales: 96
Duplicados de IDPERSONA: 0

Porcentaje de nulos por columna (top 20):
PROMEDIO_ESTUDIANTES_POR_CURSO      72.3
COBERTURA_EVALUACION                68.1
PROMEDIO_HETEROEVALUACION           68.1
CANTONORIGEN                        44.6
PROVINCIAORIGEN                     44.5
FECHANACIMIENTO                     44.1
SEXO                                44.1
PAISORIGEN                          44.1
EDAD                                44.1
ESTADOCIVIL                         44.1
DEDICACION_DOCENTE_MAS_FRECUENTE    41.8
DEDICACION_DOCENTE_ACTUAL           41.8
REGIMEN_INICIAL_DESC                19.3
TIENE_POSGRADO                       1.6
REGIMEN_ACTUAL_DESC                  0.2
N_CARGOS_DISTINTOS                   0.1
CARGO_ACTUAL                         0.1
CARGO_MAS_FRECUENTE                  0.1
MULTIPLES_CARGOS_MISMO_ANIO          0.1
N_UNIDADES_DISTINTAS                 0.1
dtype: float64


In [23]:
guardar_tabla(dataset_personas_integrado, "dataset_personas_integrado.csv")
dataset_personas_integrado.head()

Guardado dataset_personas_integrado.csv: 3956 filas x 96 columnas


,IDPERSONA,ESTADOCIVIL,FECHANACIMIENTO,SEXO,PAISORIGEN,PROVINCIAORIGEN,CANTONORIGEN,EDAD,NUM_TITULACIONES,NUM_TIT_BACHILLERATO,NUM_TIT_CUARTO_NIVEL,NUM_TIT_PRIMARIA,NUM_TIT_TERCER_NIVEL,TIENE_POSGRADO,NUM_CAPACITACIONES,SUMA_DURACION_CAPACITACIONES,NUM_CAPACITACIONES_APROBACION,NUM_CAPACITACIONES_VIRTUAL,NUM_CAPACITACIONES_PRESENCIAL,NUM_CERTIFICACIONES,SUMA_DURACION_CERTIFICACIONES,NUM_PONENCIAS,SUMA_DURACION_PONENCIAS,NUM_CURSOS,NUM_PERIODOS_DOCENCIA,NUM_ASIGNACIONES_DOCENCIA,TOTAL_HORAS_DOCENCIA,TOTAL_ESTUDIANTES,PROMEDIO_ESTUDIANTES_POR_CURSO,NUM_ACTIVIDADES_POLITECNICAS,...,N_REGISTROS_HISTORIAL,N_CONTRATOS_TOTAL,N_CARGOS_DISTINTOS,CARGO_ACTUAL,CARGO_MAS_FRECUENTE,MULTIPLES_CARGOS_MISMO_ANIO,N_UNIDADES_DISTINTAS,UNIDAD_ACTUAL_NOMBRE,N_FACULTADES_DISTINTAS,PASO_POR_RECTORADO,N_REGIMENES_DISTINTOS,REGIMEN_INICIAL_DESC,REGIMEN_ACTUAL_DESC,TIPOEMPLEADO_ACTUAL_DESC,ES_DOCENTE_ADMIN_MIXTO,ANIOS_EXPERIENCIA_DOCENTE,ANIOS_EXPERIENCIA_ADMINISTRATIVO,DEDICACION_DOCENTE_ACTUAL,DEDICACION_DOCENTE_MAS_FRECUENTE,N_DEDICACIONES_DOCENTE_DISTINTAS,PROPORCION_CONTRATOS_FINALIZADOS,FECHA_PRIMER_INGRESO,FECHA_ULTIMO_PERIODO_FIN,N_PERIODOS_CONTINUOS,ANTIGUEDAD_EFECTIVA_DIAS,VIGENTE_ACTUALMENTE,ANTIGUEDAD_EFECTIVA_ANIOS,ANTIGUEDAD_CALENDARIO_DIAS,ANTIGUEDAD_CALENDARIO_ANIOS,N_REINGRESOS
0,26,C,1972-01-01,M,ECUADOR,EL ORO,PIÑAS,54.0,4.0,0.0,3.0,0.0,1.0,1.0,12.0,590.0,6.0,1.0,0.0,0.0,0.0,3.0,25.0,35.0,14.0,35.0,3164.45,639.0,18.26,74.0,...,88.0,58.0,10.0,PROFESOR TITULAR AGREGADO 3 (TC),SUBDECANO(A),True,3.0,FACULTAD DE INGENIERÍA MECÁNICA Y CIENCIAS DE ...,2.0,False,2.0,NaN,LOES - LEY ORGANICA DE EDUCACION SUPERIOR,DOCENTE,True,17.03,3.87,Tiempo Completo,Tiempo Completo,3.0,0.96,1998-05-25,2026-09-01,11.0,6206.0,True,16.99,10327.0,28.27,10.0
1,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,1.0,0.0,1.0,1.0,4.0,606.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,12.0,6.0,12.0,1211.73,166.0,13.83,32.0,...,18.0,11.0,3.0,PROFESOR HONORARIO,PROFESOR HONORARIO,True,2.0,FACULTAD DE INGENIERÍA EN CIENCIAS DE LA TIERRA,1.0,False,1.0,CONTRATO CIVIL,CONTRATO CIVIL,DOCENTE,True,5.25,0.08,Tiempo Parcial,Tiempo Completo,3.0,1.00,2015-05-04,2023-02-17,12.0,2069.0,False,5.66,2847.0,7.79,11.0
2,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,NaN,0.0,...,6.0,6.0,2.0,SERVICIOS PROFESIONALES - EJECUCIÓN DE ACTIVID...,SERVICIOS PROFESIONALES - EJECUCIÓN DE ACTIVID...,True,1.0,GERENCIA DE INFRAESTRUCTURA FÍSICA,0.0,False,1.0,CONTRATO CIVIL,CONTRATO CIVIL,ADMINISTRATIVO,False,0.00,2.53,NaN,NaN,0.0,1.00,2016-12-09,2023-04-30,5.0,924.0,False,2.53,2334.0,6.39,4.0
3,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.0,8.0,27.0,1825.58,206.0,7.63,19.0,...,11.0,7.0,2.0,PROFESOR HONORARIO,PROFESOR HONORARIO,True,1.0,FACULTAD DE INGENIERÍA EN CIENCIAS DE LA TIERRA,1.0,False,1.0,CONTRATO CIVIL,CONTRATO CIVIL,DOCENTE,True,2.65,2.35,Tiempo Parcial,Tiempo Parcial,2.0,1.00,2018-02-01,2023-12-31,7.0,1824.0,False,4.99,2160.0,5.91,6.0
4,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,0.0,1.0,0.0,0.0,14.0,126.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,NaN,0.0,...,42.0,16.0,1.0,CHOFER,CHOFER,False,1.0,DIRECCIÓN DE SERVICIOS GENERALES,0.0,False,1.0,NaN,CT - CODIGO DE TRABAJO,ADMINISTRATIVO,False,0.00,30.30,NaN,NaN,0.0,1.00,1993-05-10,2023-09-30,2.0,10980.0,False,30.06,11101.0,30.39,1.0
